# Semiconductor equity analysis

Assess common variation in NVDA, AMD, AVGO, TSM, MU, INTC,
and ASML returns, and how much remains after controlling for
the broad market.

Initial benchmark: SPY.

In [9]:
%pip install yfinance

  Using cached peewee-4.5.3-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 7.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 7.4 MB/s  0:00:01 eta 0:00:01
Using cached peewee-4.5.3-py3-none-any.whl (194 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [yfinance]/16 [curl_cffi]]
Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf

In [3]:
equity_groups = {
    "designers": ["NVDA", "AMD", "AVGO"],
    "manufacturers": ["TSM", "MU", "INTC"],
    "equipment": ["ASML"],
}

equity_tickers = [
    ticker
    for tickers in equity_groups.values()
    for ticker in tickers
]

benchmark_ticker = "SPY"

In [20]:
tickers = equity_tickers + [benchmark_ticker]

prices = yf.download(
    tickers=tickers,
    start="2010-01-01",
    end="2023-01-01",
    interval="1d",
    auto_adjust=False,
    actions=True,
    keepna=True,
    repair=False,
)

[*********************100%***********************]  8 of 8 completed


In [14]:
prices.shape

(2014, 65)

In [21]:
prices.head()

Price      Adj Close                                                       \
Ticker           AMD       ASML      AVGO       INTC         MU      NVDA   
Date                                                                        
2010-01-04      9.70  32.228188  1.321281  13.519423  10.578575  0.422817   
2010-01-05      9.71  32.479244  1.331089  13.512953  10.890566  0.428991   
2010-01-06      9.57  32.776794  1.341597  13.467627  10.939317  0.431736   
2010-01-07      9.47  31.865549  1.333190  13.338132  10.568826  0.423275   
2010-01-08      9.43  31.103069  1.342999  13.487051  10.822319  0.424189   

Price                           Capital Gains Close  ... Stock Splits       \
Ticker            SPY       TSM           SPY   AMD  ...          SPY  TSM   
Date                                                 ...                     
2010-01-04  84.368965  7.199557           0.0  9.70  ...          0.0  0.0   
2010-01-05  84.592316  7.168467           0.0  9.71  ...          0.0  0.0   
2010-01-06  84.651878  7.143602           0.0  9.57  ...          0.0  0.0   
2010-01-07  85.009224  6.907347           0.0  9.47  ...          0.0  0.0   
2010-01-08  85.292091  6.901131           0.0  9.43  ...          0.0  0.0   

Price         Volume                                                   \
Ticker           AMD     ASML     AVGO      INTC        MU       NVDA   
Date                                                                    
2010-01-04  18748700  1390235  5387000  47800900  34412700  800204000   
2010-01-05  22145700  1094016  3011000  52357700  60659000  728648000   
2010-01-06  18643400  1051204  5331000  40037400  32060400  649168000   
2010-01-07  26806800  2057440  2718000  54041500  46461800  547792000   
2010-01-08  13752800  3027178  2899000  48234700  32433400  478168000   

Price                            
Ticker            SPY       TSM  
Date                             
2010-01-04  118944600   8096400  
2010-01-05  111579900  14375900  
2010-01-06  116074400  13608400  
2010-01-07  131091100  27346600  
2010-01-08  126402800  16895300  

[5 rows x 65 columns]

In [22]:
adjusted_close = prices["Adj Close"].copy()

data_check = pd.DataFrame({
    "first_date": adjusted_close.apply(
        lambda column: column.first_valid_index()
    ),
    "last_date": adjusted_close.apply(
        lambda column: column.last_valid_index()
    ),
    "missing_prices": adjusted_close.isna().sum(),
})

display(data_check)

print("Dates in order:", adjusted_close.index.is_monotonic_increasing)
print("Duplicate dates:", adjusted_close.index.has_duplicates)

,first_date,last_date,missing_prices
Ticker,,,
AMD,2010-01-04,2022-12-30,0
ASML,2010-01-04,2022-12-30,0
AVGO,2010-01-04,2022-12-30,0
INTC,2010-01-04,2022-12-30,0
MU,2010-01-04,2022-12-30,0
NVDA,2010-01-04,2022-12-30,0
SPY,2010-01-04,2022-12-30,0
TSM,2010-01-04,2022-12-30,0


Dates in order: True
Duplicate dates: False
